<a href="https://colab.research.google.com/github/Termuni/IVS_Study/blob/Python_Study/Camera_calibration_GT_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 최신 OpenCV 설치
!pip install --upgrade opencv-python opencv-python-headless

캘리브레이션 과정에서 사용할 Numpy와 OpenCV 불러오기

Calibration 데이터 다운로드

In [ ]:
import cv2
import numpy as np
import glob
import matplotlib.pyplot as plt
import os


# 다운로드한 파일을 저장할 디렉토리 생성
if not os.path.exists('calibration_images'):
    os.makedirs('calibration_images')

# 각각의 이미지를 개별적으로 다운로드하여 calibration_images 폴더에 저장
# 10번 이미지 오류로 제외함
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left01.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left02.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left03.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left04.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left05.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left06.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left07.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left08.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left09.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left11.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left12.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left13.jpg
!wget -q -P calibration_images https://github.com/opencv/opencv/raw/master/samples/data/left14.jpg

**이미지 로드 및 시각화**

OpenCV와 Matplotlib을 사용하여 이미지를 읽고 시각화

In [ ]:
# 첫 번째 이미지 파일 경로 설정
image_path = 'calibration_images/left01.jpg'

# OpenCV를 사용하여 이미지 읽기
img = cv2.imread(image_path)

# 이미지가 제대로 로드되었는지 확인
if img is None:
    print(f"Error: Unable to load image from {image_path}")
else:
    print(f"Successfully loaded image from {image_path}")

    # OpenCV는 기본적으로 BGR 형식으로 이미지를 로드하므로, RGB 형식으로 변환
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 이미지 시각화
    plt.figure(figsize=(8, 6))
    plt.imshow(img_rgb)
    plt.title('Calibration Image - left01.jpg')
    plt.axis('off')  # 축을 숨김
    plt.show()

**이미지의 기본 특성 출력**

이미지의 기본적인 특성, 즉 크기, 채널 수, 데이터 타입 등을 출력

In [ ]:
# 이미지의 크기와 채널 수 확인
height, width, channels = img.shape
print(f"Image dimensions (Height, Width, Channels): {height} x {width} x {channels}")

# 이미지의 데이터 타입 확인
print(f"Image data type: {img.dtype}")

# 이미지의 최소 및 최대 픽셀 값 확인 (픽셀 값은 0~255 범위 내)
min_val, max_val = img.min(), img.max()
print(f"Minimum pixel value: {min_val}")
print(f"Maximum pixel value: {max_val}")

**이미지 분석**

해당 이미지의 경우 흑백이므로, (r,g,b)에서 서로 값이 동일한 값으로 구성됨.

In [ ]:
# 이미지를 (Height*Width) x Channels 형태로 변환
reshaped_img = img.reshape(-1, 3)

# 고유한 RGB 조합의 수 확인
unique_colors = np.unique(reshaped_img, axis=0)

print(f"Unique RGB combinations: {len(unique_colors)}")
print("Sample of unique colors:")
len(unique_colors)  # 최대 10개의 고유 RGB 값 샘플 출력

체스보드는 정밀한 사각형 그리드 패턴을 가지는데, 이때 캘리브레이션을 위해 체스보드의 내부 코너를 사용

In [ ]:
# 체커보드의 내부 코너 수 설정
board_size = (8, 6)

3D 공간에서 체커보드의 코너는 (x, y, z) 좌표를 가짐.

캘리브레이션 과정에서는 체커보드가 평평한 표면에 놓여 있다고 가정하여 모든 z 좌표를 0으로 설정합니다.

In [ ]:
# 체커보드의 3D 포인트 준비
objp = np.zeros((board_size[0] * board_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:board_size[0], 0:board_size[1]].T.reshape(-1, 2)

# 3D 포인트 예시
print(f"3D 포인트의 size : {objp.shape}")

print(f"3D 포인트 예시 : {objp[4]}")

 'objpoints'는 모든 이미지에 대해 동일한 체스보드의 3D 좌표를 저장

 'imgpoints'는 각 이미지에서 검출된 체스보드 코너의 2D 좌표를 저장

 캘리브레이션에 사용할 이미지 로드

In [ ]:
# 3D 포인트와 2D 이미지 포인트 저장을 위한 배열 생성
objpoints = []  # 3D 포인트 (실제 공간)
imgpoints = []  # 2D 포인트 (이미지 평면)

# 캘리브레이션 이미지가 저장된 경로 설정
images = glob.glob('calibration_images/*.jpg')

print(images)

if len(images) == 0:
    print("Error: No images found. Please check the path or upload images.")
else:
    print(f"Found {len(images)} images.")

각 이미지에 대해 체스보드 코너를 검출하고, 검출된 코너를 저장

cv2.findChessboardCorners : 그레이 스케일 이미지(channel dimension이 1인 이미지)에서 체커보드의 코너를 찾는 함수로 Harris Corner Detector를 이용하여 찾음

In [ ]:
# 이미지에서 체커보드 코너 찾기
for idx, fname in enumerate(images):
    img = cv2.imread(fname)
    if img is None:
        print(f"Error: Unable to load image {fname}. Skipping this file.")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 체커보드 코너 찾기
    ret, corners = cv2.findChessboardCorners(gray, board_size, None)

    if ret:
        objpoints.append(objp)
        imgpoints.append(corners)

        # 코너 그리기 및 이미지 출력 (첫번째, 두번째 이미지만 시각화)
        if idx in [0,1]:
            cv2.drawChessboardCorners(img, board_size, corners, ret)
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.show()
    else:
        print(f"Warning: Chessboard corners not found for image {fname}.")


**카메라 캘리브레이션 수행**

체스보드 코너가 검출된 이미지를 사용하여 카메라의 내부 및 외부 파라미터를 계산

여기서 외부 파라미터는 체커보드 기준 3D 좌표게와 현재 이미지가 찍힌 2D 좌표계 사이의 파라미터를 의미함. 따라서 각 이미지마다 생성됨.

In [ ]:
# 카메라 캘리브레이션 수행
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

print("Camera intrinsic matrix:")
print(mtx)
print("\nDistortion coefficients:")
print(dist)
print("\n2번째 이미지 rotation value:")
print(rvecs[1])
print("\n2번째 이미지 translation value:")
print(tvecs[1])

**2번째 이미지에 대한 외부파라미터 생성**

In [ ]:
index = 1

rvec = rvecs[index]
tvec = tvecs[index]

# 로드리게스 변환을 사용하여 회전 벡터를 회전 행렬로 변환
rotation_matrix, _ = cv2.Rodrigues(rvec)

# 회전 행렬과 변환 벡터를 결합하여 3x4 외부 행렬 생성
extrinsic_matrix = np.hstack((rotation_matrix, tvec))

print(f"Image: {images[index]}")
print(f"\nRotation matrix (R):\n{rotation_matrix}")
print(f"\nTranslation vector (t):\n{tvec}")
print(f"\nExtrinsic matrix [R|t]:\n{extrinsic_matrix}")

**왜곡 보정 및 결과 확인**

2번째 이미지에 대하여, 지금까지 얻은 캘리브레이션 결과를 사용해 왜곡된 이미지를 보정

카메라 렌즈의 비선형 왜곡을 제거하는 과정

cv2.getOptimalNewCameraMatrix : 왜곡된 이미지를 잘라내지 않으면서 왜곡을 보정하기 위한 새로운 카메라 행렬을 계산

cv2.undistort : 왜곡된 이미지를 보정하여 왜곡이 없는 이미지를 생성

In [ ]:
# 예시 이미지에서 왜곡 보정 수행
img = cv2.imread(images[1])
h, w = img.shape[:2]

# 새로운 카메라 행렬 계산 (ROI 포함)
# roi: 왜곡 보정의 유효한 영역을 나타냄
newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), 1, (w, h))

# 왜곡 보정
dst = cv2.undistort(img, mtx, dist, None, newcameramtx)

# 결과 이미지 보여주기
plt.figure(figsize=(12, 6))
plt.subplot(121), plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), plt.title('Original Image')
plt.subplot(122), plt.imshow(cv2.cvtColor(dst, cv2.COLOR_BGR2RGB)), plt.title('Undistorted Image')
plt.show()

print(roi)

In [ ]:
# ROI에 기반한 사각형 좌표 추출
x, y, w, h = roi

dst_with_roi = dst.copy()
cv2.rectangle(dst_with_roi, (x, y), (x + w, y + h), (0, 0, 255), 2) # 빨간색 (BGR), 두께 2


# 결과 이미지 시각화
plt.figure(figsize=(12, 6))

# 원본 이미지와 왜곡 보정된 이미지에 표시된 ROI
plt.subplot(121)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title('Original Image with ROI')
plt.axis('off')

# 왜곡 보정된 이미지와 ROI 영역 표시
plt.subplot(122)
plt.imshow(cv2.cvtColor(dst_with_roi, cv2.COLOR_BGR2RGB))
plt.title('Undistorted Image with ROI')
plt.axis('off')

plt.show()

**재투영 오차 계산**

재투영 오차는 캘리브레이션의 정확도를 평가하는 지표로 캘리브레이션 정확도 향상을 목표로 함

3D 객체 포인트를 이미지에 재투영 이후 실제 2D 포인트와 비교하여 에러 산출

한계 : imgpoints 역시 실제 값이 아닌 코너 디텍터에서 얻은 값이므로, 코너 디텍터의 성능이 높아질수록 정확한 평가 가능.

In [ ]:
# 재투영 에러 계산
mean_error = 0
for i in range(len(objpoints)):
    # 3D 객체 포인트를 카메라 파라미터를 사용해 2D 이미지 평면으로 재투영
    imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)

    # 실제 2D 이미지 포인트와 재투영된 2D 포인트 간의 에러 계산
    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)

    # 에러 합산
    mean_error += error

# 평균 재투영 에러 계산
print(f"Total reprojection error: {mean_error / len(objpoints)}")